In [ ]:
import os, time, random, glob, warnings, pickle, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
import timm
import torchvision.models as tvmodels
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, confusion_matrix, classification_report
from scipy.spatial.distance import cdist
from PIL import Image
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
%matplotlib inline

DATA_DIR     = r"C:\Users\murat akkaya\Desktop\colon_4_class"
SWIN_WEIGHTS = r"C:\Users\murat akkaya\Desktop\swin_base_pretrained.pth"
REPORT_DIR   = r"E:\CL_DataSize_Ablation_v3"
HARDNESS_DIR = os.path.join(REPORT_DIR, 'hardness')
HARDNESS_CSV = os.path.join(HARDNESS_DIR, 'hardness_scores.csv')

COMPUTE_HARDNESS_FRESH = True
RESNET_EPOCHS      = 15
RESNET_LR          = 1e-4
RESNET_WD          = 1e-4
RESNET_BATCH       = 64
RESNET_FEAT_DIM    = 2048
HARDNESS_FOLDS     = 5
SAVE_SCORER_MODELS = True
HARDNESS_SEED      = 123

N_SPLITS     = 5
BATCH_SIZE   = 64
NUM_EPOCHS   = 30
LR           = 1e-4
WEIGHT_DECAY = 1e-4
SEED         = 42
NUM_WORKERS  = 0
EMBED_DIM    = 1024
USE_AMP      = True
IMG_SIZE     = 224
CACHE_IMAGES = True

CL_START_RATIO = 0.33
CL_RAMP_EPOCHS = 25
COMPETENCE_FN  = 'linear'

DATA_FRACTIONS = [0.10, 0.25, 0.50, 1.00]

SAVE_FOLDS      = []
SAVE_LAST_MODEL = False

CLASSES = ["adenomatous", "cancer", "inflamed", "normal"]
NUM_CLASSES = len(CLASSES)

for d in ['', 'models', 'histories', 'reports', 'hardness', 'partial']:
    os.makedirs(os.path.join(REPORT_DIR, d), exist_ok=True)

torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == 'cuda':
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Competence: {COMPETENCE_FN} | Fractions: {DATA_FRACTIONS}")

In [ ]:
class HistopathDataset(Dataset):
    def __init__(self, root_dir, classes, transform=None, cache=True, verbose=True):
        self.transform = transform
        self.cache = cache
        self._cache = {}
        self.samples = []
        self.class_to_idx = {c: i for i, c in enumerate(classes)}
        for cls in classes:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for ext in ['*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp']:
                for p in sorted(glob.glob(os.path.join(cls_dir, ext))):
                    self.samples.append((p, self.class_to_idx[cls]))
        if verbose:
            print(f"Loaded {len(self.samples)} samples")
            for cls in classes:
                n = sum(1 for _, l in self.samples if l == self.class_to_idx[cls])
                print(f"  {cls}: {n}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        if self.cache and idx in self._cache:
            img = self._cache[idx]
        else:
            img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
            if self.cache:
                self._cache[idx] = img
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = HistopathDataset(DATA_DIR, CLASSES, train_transform, cache=CACHE_IMAGES)
eval_dataset  = HistopathDataset(DATA_DIR, CLASSES, eval_transform, cache=CACHE_IMAGES, verbose=False)
eval_dataset._cache = train_dataset._cache

if CACHE_IMAGES:
    print("Caching images (shared)...")
    for i in tqdm(range(len(train_dataset))):
        train_dataset[i]
    print("Cache ready.")

all_indices = np.arange(len(train_dataset))
all_targets = np.array([s[1] for s in train_dataset.samples])
all_paths   = [s[0] for s in train_dataset.samples]
all_names   = [os.path.basename(p) for p in all_paths]
N_FULL = len(train_dataset)
print(f"Full dataset: {N_FULL}")

In [ ]:
class FeatureExtractorModel(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        weights = tvmodels.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        base = tvmodels.resnet50(weights=weights)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Linear(RESNET_FEAT_DIM, num_classes)

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.classifier(feat), feat


def train_scorer(train_idx):
    model = FeatureExtractorModel(NUM_CLASSES, pretrained=True).to(device)
    opt = optim.AdamW(model.parameters(), lr=RESNET_LR, weight_decay=RESNET_WD)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=RESNET_EPOCHS)
    scaler = GradScaler('cuda', enabled=USE_AMP)

    loader = DataLoader(Subset(train_dataset, train_idx), batch_size=RESNET_BATCH,
                        shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

    for epoch in range(1, RESNET_EPOCHS + 1):
        model.train()
        run_loss, correct, total = 0.0, 0, 0
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=USE_AMP):
                logits, _ = model(imgs)
                loss = F.cross_entropy(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item()
            correct += logits.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
        sched.step()
        if epoch % 5 == 0 or epoch == 1:
            print(f"    ep{epoch:2d}/{RESNET_EPOCHS} | loss {run_loss/len(loader):.4f} "
                  f"| train acc {100.0*correct/total:.2f}%")

    del opt, sched, scaler
    return model


def extract_features_only(model, indices):
    loader = DataLoader(Subset(eval_dataset, indices), batch_size=RESNET_BATCH,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    Fe, Y = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                _, feats = model(imgs)
            Fe.append(feats.float().cpu().numpy())
            Y.append(labels.numpy())
    return np.vstack(Fe), np.concatenate(Y)


def score_samples(model, indices):
    loader = DataLoader(Subset(eval_dataset, indices), batch_size=RESNET_BATCH,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    model.eval()
    L, C, P, Fe, Y = [], [], [], [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            lab_dev = labels.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                logits, feats = model(imgs)
            logits = logits.float()
            L.append(F.cross_entropy(logits, lab_dev, reduction='none').cpu().numpy())
            probs = torch.softmax(logits, dim=1)
            mx, pr = probs.max(dim=1)
            C.append((1.0 - mx).cpu().numpy())
            P.append(pr.cpu().numpy())
            Fe.append(feats.float().cpu().numpy())
            Y.append(labels.numpy())
    return (np.concatenate(L), np.concatenate(C), np.concatenate(P),
            np.vstack(Fe), np.concatenate(Y))

print("Scorer functions ready.")

In [ ]:
if COMPUTE_HARDNESS_FRESH:
    skf_h = StratifiedKFold(n_splits=HARDNESS_FOLDS, shuffle=True, random_state=HARDNESS_SEED)
    h_folds = list(skf_h.split(np.zeros(N_FULL), all_targets))

    oof_loss    = np.zeros(N_FULL)
    oof_conf    = np.zeros(N_FULL)
    oof_pred    = np.zeros(N_FULL, dtype=int)
    oof_distown = np.zeros(N_FULL)
    oof_distoth = np.zeros(N_FULL)
    oof_fold    = np.zeros(N_FULL, dtype=int)
    features    = np.zeros((N_FULL, RESNET_FEAT_DIM), dtype=np.float32)

    fold_acc, fold_f1 = [], []
    t0 = time.time()

    for fi, (tr_idx, te_idx) in enumerate(h_folds, start=1):
        print(f"\n{'='*58}")
        print(f"  Hardness scorer fold {fi}/{HARDNESS_FOLDS} "
              f"({len(tr_idx)} train / {len(te_idx)} scored)")
        print(f"{'='*58}")

        model = train_scorer(tr_idx)

        tr_feats, tr_labels = extract_features_only(model, tr_idx)
        centroid_matrix = np.array([tr_feats[tr_labels == c].mean(axis=0)
                                    for c in range(NUM_CLASSES)])
        del tr_feats
        gc.collect()

        l, c, p, fe, y = score_samples(model, te_idx)

        d_all = cdist(fe, centroid_matrix, metric='euclidean')
        d_own = d_all[np.arange(len(y)), y]
        masked = d_all.copy()
        masked[np.arange(len(y)), y] = np.inf
        d_oth = masked.min(axis=1)

        oof_loss[te_idx]    = l
        oof_conf[te_idx]    = c
        oof_pred[te_idx]    = p
        oof_distown[te_idx] = d_own
        oof_distoth[te_idx] = d_oth
        oof_fold[te_idx]    = fi
        features[te_idx]    = fe

        acc = 100.0 * (p == y).mean()
        f1 = f1_score(y, p, average='macro') * 100
        fold_acc.append(acc)
        fold_f1.append(f1)
        print(f"  held-out: acc {acc:.2f}% | macro F1 {f1:.2f}% | mean loss {l.mean():.4f}")

        if SAVE_SCORER_MODELS:
            torch.save({'state_dict': {k: v.cpu() for k, v in model.state_dict().items()},
                        'fold': fi, 'train_idx': tr_idx, 'test_idx': te_idx,
                        'held_out_acc': acc, 'held_out_f1': f1,
                        'arch': 'resnet50', 'class_names': CLASSES},
                       os.path.join(HARDNESS_DIR, f'resnet50_scorer_fold{fi}.pth'))

        del model, l, c, p, fe, y, d_all, masked, centroid_matrix
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\nScorer held-out accuracy: {np.mean(fold_acc):.2f} +/- {np.std(fold_acc):.2f}%")
    print(f"Scorer held-out macro F1: {np.mean(fold_f1):.2f} +/- {np.std(fold_f1):.2f}%")
    print(f"Total scoring time: {(time.time()-t0)/60:.1f} min")

    def norm_within_folds(raw, folds, invert_sigmoid=False):
        out = np.zeros_like(raw, dtype=float)
        for fi in range(1, HARDNESS_FOLDS + 1):
            m = folds == fi
            v = raw[m]
            if invert_sigmoid:
                s = v.std() if v.std() > 0 else 1.0
                out[m] = 1.0 / (1.0 + np.exp(2.0 * v / s))
            else:
                out[m] = (v - v.min()) / (v.max() - v.min() + 1e-8)
        return out

    margin_raw    = oof_distoth - oof_distown
    loss_hard     = norm_within_folds(oof_loss, oof_fold)
    conf_hard     = oof_conf
    centroid_hard = norm_within_folds(oof_distown, oof_fold)
    boundary_hard = norm_within_folds(margin_raw, oof_fold, invert_sigmoid=True)
    composite     = 0.25 * (loss_hard + conf_hard + centroid_hard + boundary_hard)

    correct_mask = (oof_pred == all_targets)
    cls_of = {i: c for i, c in enumerate(CLASSES)}
    hard_df = pd.DataFrame({
        'image_path': all_paths,
        'filename': all_names,
        'true_label': all_targets,
        'true_class': [cls_of[l] for l in all_targets],
        'predicted_label': oof_pred,
        'predicted_class': [cls_of[p] for p in oof_pred],
        'correct': correct_mask.astype(int),
        'scorer_fold': oof_fold,
        'loss_raw': oof_loss,
        'confidence_raw': 1.0 - conf_hard,
        'centroid_distance_raw': oof_distown,
        'dist_to_nearest_other': oof_distoth,
        'margin_raw': margin_raw,
        'hardness_loss': loss_hard,
        'hardness_confidence': conf_hard,
        'hardness_centroid': centroid_hard,
        'hardness_boundary': boundary_hard,
        'hardness_composite': composite,
    })
    for col in ['loss', 'confidence', 'centroid', 'boundary', 'composite']:
        hard_df['group_' + col] = pd.qcut(hard_df['hardness_' + col], 3,
                                          labels=['easy', 'medium', 'hard'], duplicates='drop')

    hard_df.to_csv(HARDNESS_CSV, index=False)
    np.save(os.path.join(HARDNESS_DIR, 'features.npy'), features)
    np.save(os.path.join(HARDNESS_DIR, 'labels.npy'), all_targets)
    print(f"\nSaved: {HARDNESS_CSV}  {hard_df.shape}")
    print(f"Out-of-fold accuracy: {correct_mask.mean()*100:.2f}%")
    gc.collect()
else:
    hard_df = pd.read_csv(HARDNESS_CSV)
    features = None
    print(f"Loaded: {HARDNESS_CSV}  {hard_df.shape}")

In [ ]:
hcols = ['hardness_loss', 'hardness_confidence', 'hardness_centroid',
         'hardness_boundary', 'hardness_composite']
short = ['Loss', 'Confidence', 'Centroid', 'Boundary', 'Composite']

corr = hard_df[hcols].corr()
print(corr.round(3).to_string())
print()
print(hard_df.groupby('true_class')[hcols].mean().round(4).to_string())

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr.values, cmap='RdYlBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(short)))
ax.set_yticks(range(len(short)))
ax.set_xticklabels(short, rotation=45, ha='right')
ax.set_yticklabels(short)
for i in range(len(short)):
    for j in range(len(short)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha='center', va='center',
                color='white' if abs(corr.values[i, j]) > 0.5 else 'black', fontsize=11)
plt.colorbar(im, ax=ax, shrink=0.8)
plt.title('Hardness metric correlation')
plt.tight_layout()
plt.savefig(os.path.join(HARDNESS_DIR, 'correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if features is None:
    fpath = os.path.join(HARDNESS_DIR, 'features.npy')
    features = np.load(fpath) if os.path.exists(fpath) else None

if features is not None:
    pca = PCA(n_components=2)
    pcs = pca.fit_transform(StandardScaler().fit_transform(features))
    ev = pca.explained_variance_ratio_ * 100
    print(f"PC1 {ev[0]:.1f}% | PC2 {ev[1]:.1f}% | total {ev.sum():.1f}%")

    labels_arr = hard_df['true_label'].values
    class_colors = ['#0072B2', '#D55E00', '#009E73', '#CC79A7']
    grp_colors = ['#2ecc71', '#f39c12', '#e74c3c']
    grp_names = ['Easy', 'Medium', 'Hard']
    panels = [('loss', 'Loss hardness'), ('centroid', 'Centroid distance hardness'),
              ('boundary', 'Boundary proximity hardness')]

    fig, axes = plt.subplots(2, 2, figsize=(15, 13))
    axes = axes.flatten()

    for ci, cname in enumerate(CLASSES):
        m = labels_arr == ci
        axes[0].scatter(pcs[m, 0], pcs[m, 1], c=class_colors[ci], label=cname, alpha=0.5, s=7)
    axes[0].set_title('(a) Class label', fontsize=13, fontweight='bold')
    axes[0].legend(markerscale=3, fontsize=9)

    for k, (key, title) in enumerate(panels, start=1):
        scores = hard_df['hardness_' + key].values
        t1, t2 = np.percentile(scores, [33.3, 66.6])
        grp = np.zeros(len(scores), dtype=int)
        grp[scores > t1] = 1
        grp[scores > t2] = 2
        for g in range(3):
            m = grp == g
            axes[k].scatter(pcs[m, 0], pcs[m, 1], c=grp_colors[g],
                            label=f'{grp_names[g]} ({m.sum()})', alpha=0.5, s=7)
        axes[k].set_title(f'({chr(97+k)}) {title}', fontsize=13, fontweight='bold')
        axes[k].legend(markerscale=3, fontsize=9)

    for ax in axes:
        ax.set_xlabel(f'PC1 ({ev[0]:.1f}%)')
        ax.set_ylabel(f'PC2 ({ev[1]:.1f}%)')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(HARDNESS_DIR, 'pca_hardness_panels.png'), dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
def scores_from_df(df, column, names):
    base = df['filename'].astype(str).apply(lambda x: os.path.splitext(x)[0])
    mapping = dict(zip(base, df[column]))
    med = df[column].median()
    out = np.array([mapping.get(os.path.splitext(n)[0], med) for n in names])
    matched = sum(1 for n in names if os.path.splitext(n)[0] in mapping)
    print(f"  {column:22s} matched {matched}/{len(names)} | mean {out.mean():.4f}")
    return out

hardness_loss     = scores_from_df(hard_df, 'hardness_loss', all_names)
hardness_boundary = scores_from_df(hard_df, 'hardness_boundary', all_names)
hardness_centroid = scores_from_df(hard_df, 'hardness_centroid', all_names)

np.random.seed(SEED)
hardness_random = np.random.rand(N_FULL)

HARDNESS_SCORES = {
    'Loss': hardness_loss, 'Boundary': hardness_boundary,
    'Centroid': hardness_centroid, 'Random': hardness_random,
}

STRATEGIES = {}
for h in ['Loss', 'Boundary', 'Centroid']:
    STRATEGIES[f'{h}_E2H'] = {'hardness': h, 'direction': 'e2h'}
    STRATEGIES[f'{h}_H2E'] = {'hardness': h, 'direction': 'h2e'}
STRATEGIES['Random'] = {'hardness': 'Random', 'direction': 'e2h'}

print(f"{len(STRATEGIES)} strategies: {list(STRATEGIES.keys())}")

In [ ]:
def get_stratified_subset(all_indices, all_targets, fraction, seed=42):
    if fraction >= 1.0:
        subset_idx = all_indices.copy()
    else:
        _, subset_idx = train_test_split(all_indices, test_size=fraction,
                                         stratify=all_targets, random_state=seed)
        subset_idx = np.sort(subset_idx)
    labs = all_targets[subset_idx]
    print(f"  Subset: {len(subset_idx)} samples ({fraction*100:.0f}%)")
    for ci, cname in enumerate(CLASSES):
        print(f"    {cname}: {(labs == ci).sum()}")
    return subset_idx

SUBSETS = {}
for frac in DATA_FRACTIONS:
    print(f"\n--- {frac*100:.0f}% subset ---")
    SUBSETS[frac] = get_stratified_subset(all_indices, all_targets, frac, SEED)

n_ckpt = (1 if SAVE_FOLDS else 0) + (1 if (SAVE_FOLDS and SAVE_LAST_MODEL) else 0)
n_models = len(STRATEGIES) * len(DATA_FRACTIONS) * len(SAVE_FOLDS) * max(n_ckpt, 0)
n_runs = len(STRATEGIES) * len(DATA_FRACTIONS) * N_SPLITS

print(f"\n{'='*55}")
print(f"  Training runs : {n_runs}")
print(f"  Models saved  : {n_models}  (~{n_models*0.35:.1f} GB)")
print(f"{'='*55}")

In [ ]:
if not os.path.exists(SWIN_WEIGHTS):
    _tmp = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=0)
    torch.save(_tmp.state_dict(), SWIN_WEIGHTS)
    del _tmp
    torch.cuda.empty_cache()

def create_model():
    backbone = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=0)
    backbone.load_state_dict(torch.load(SWIN_WEIGHTS, weights_only=True))
    return nn.Sequential(backbone, nn.Linear(EMBED_DIM, NUM_CLASSES)).to(device)

def competence(t, T=CL_RAMP_EPOCHS, c0=CL_START_RATIO):
    fn = COMPETENCE_FN
    if fn == 'linear':
        c = c0 + (1.0 - c0) * (t / T)
    else:
        p = 2 if fn == 'sqrt' else int(fn[4:])
        c = (t * (1.0 - c0**p) / T + c0**p) ** (1.0 / p)
    return min(c, 1.0)

def get_curriculum_indices(train_idx, hardness_scores, epoch, direction='e2h'):
    ratio = competence(epoch - 1)
    hs = hardness_scores[train_idx]
    order = np.argsort(hs) if direction == 'e2h' else np.argsort(hs)[::-1]
    n = max(int(len(train_idx) * ratio), min(BATCH_SIZE, len(train_idx)))
    return train_idx[order[:n]], n

def validate(model, loader):
    model.eval()
    preds, labs, total_loss = [], [], 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                out = model(imgs)
                total_loss += F.cross_entropy(out, labels).item()
            preds.extend(out.argmax(1).cpu().numpy())
            labs.extend(labels.cpu().numpy())
    preds, labs = np.array(preds), np.array(labs)
    return (total_loss / len(loader), 100 * (preds == labs).mean(),
            f1_score(labs, preds, average='macro') * 100, preds, labs)

def save_checkpoint(state_dict, strategy, frac, fold, epoch, val_f1, tag,
                    train_idx, test_idx, subset_idx):
    fname = f"{strategy}_frac{int(frac*100):03d}_fold{fold}_{tag}.pth"
    path = os.path.join(REPORT_DIR, 'models', fname)
    torch.save({
        'state_dict': state_dict,
        'strategy': strategy, 'fraction': frac, 'fold': fold,
        'epoch': epoch, 'val_f1': val_f1, 'tag': tag,
        'class_names': CLASSES,
        'train_idx': np.asarray(train_idx),
        'test_idx': np.asarray(test_idx),
        'subset_idx': np.asarray(subset_idx),
        'img_size': IMG_SIZE, 'embed_dim': EMBED_DIM,
        'arch': 'swin_base_patch4_window7_224',
        'competence_fn': COMPETENCE_FN,
        'hardness_csv': HARDNESS_CSV,
    }, path)
    return path

for e in [1, 5, 15, 25, 26, 30]:
    print(f"epoch {e:2d}: {competence(e-1)*100:5.1f}%")

In [ ]:
def train_fold(strategy_name, hardness_scores, direction,
               train_idx, test_idx, subset_idx, fold, frac, frac_label):
    model = create_model()
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
    scaler = GradScaler('cuda', enabled=USE_AMP)

    test_loader = DataLoader(Subset(eval_dataset, test_idx), batch_size=BATCH_SIZE,
                             shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    n_total = len(train_idx)
    best_f1, best_epoch, best_time, best_state = 0.0, 0, 0.0, None
    saved_paths = {}
    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [],
               'val_acc': [], 'samples_used': []}
    do_save = fold in SAVE_FOLDS
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        cl_idx, n_used = get_curriculum_indices(train_idx, hardness_scores, epoch, direction)
        train_loader = DataLoader(Subset(train_dataset, cl_idx), batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

        model.train()
        run_loss = 0.0
        ep_preds, ep_labels = [], []
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with autocast('cuda', enabled=USE_AMP):
                out = model(imgs)
                loss = F.cross_entropy(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.item()
            ep_preds.extend(out.argmax(1).detach().cpu().numpy())
            ep_labels.extend(labels.detach().cpu().numpy())
        scheduler.step()

        train_loss = run_loss / len(train_loader)
        train_f1 = f1_score(ep_labels, ep_preds, average='macro') * 100
        val_loss, val_acc, val_f1, _, _ = validate(model, test_loader)

        history['train_loss'].append(train_loss)
        history['train_f1'].append(train_f1)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_acc'].append(val_acc)
        history['samples_used'].append(n_used)

        if val_f1 > best_f1:
            best_f1, best_epoch = val_f1, epoch
            best_time = time.time() - start_time
            if do_save:
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{frac_label}] [{strategy_name}] F{fold} Ep{epoch:2d} | "
                  f"CL: {n_used}/{n_total} | TF1: {train_f1:.1f}% | VF1: {val_f1:.1f}%")

    total_time = time.time() - start_time

    _, last_acc, last_f1, last_preds, last_labels = validate(model, test_loader)
    last_per_class = (f1_score(last_labels, last_preds, average=None) * 100).tolist()
    last_cm = confusion_matrix(last_labels, last_preds)

    if do_save:
        if best_state is not None:
            saved_paths['best'] = save_checkpoint(best_state, strategy_name, frac, fold,
                                                  best_epoch, best_f1, 'best',
                                                  train_idx, test_idx, subset_idx)
        if SAVE_LAST_MODEL:
            last_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            saved_paths['last'] = save_checkpoint(last_state, strategy_name, frac, fold,
                                                  NUM_EPOCHS, last_f1, 'last',
                                                  train_idx, test_idx, subset_idx)
            del last_state

        rep = classification_report(last_labels, last_preds, target_names=CLASSES, zero_division=0)
        rname = f"{strategy_name}_frac{int(frac*100):03d}_fold{fold}.txt"
        with open(os.path.join(REPORT_DIR, 'reports', rname), 'w') as f:
            f.write(f"Strategy: {strategy_name}\nFraction: {frac}\nFold: {fold}\n")
            f.write(f"Competence: {COMPETENCE_FN}\n")
            f.write(f"Best epoch: {best_epoch} (F1 {best_f1:.2f}%)\n")
            f.write(f"Last epoch: {NUM_EPOCHS} (F1 {last_f1:.2f}%)\n\n")
            f.write(rep)

    print(f"  Best: ep{best_epoch} VF1={best_f1:.2f}% ({best_time:.0f}s) | "
          f"Last: VF1={last_f1:.2f}% ({total_time:.0f}s)")

    del model, optimizer, scheduler, scaler, best_state
    torch.cuda.empty_cache()
    gc.collect()

    return {'history': history, 'best_f1': best_f1, 'best_epoch': best_epoch,
            'best_time': best_time, 'last_f1': last_f1, 'last_acc': last_acc,
            'last_per_class': last_per_class, 'last_cm': last_cm,
            'total_time': total_time, 'saved_paths': saved_paths}

In [ ]:
strategies_to_run = list(STRATEGIES.keys())
PARTIAL_DIR = os.path.join(REPORT_DIR, 'partial')
os.makedirs(PARTIAL_DIR, exist_ok=True)

def partial_path(frac, strat):
    return os.path.join(PARTIAL_DIR, f"{strat}_frac{int(frac*100):03d}.pkl")

all_fraction_results = {}
grand_start = time.time()

for frac in DATA_FRACTIONS:
    frac_label = f"{frac*100:.0f}%"
    subset_idx = SUBSETS[frac]
    subset_targets = all_targets[subset_idx]

    print(f"\n{'#'*70}")
    print(f"  DATA FRACTION: {frac_label} ({len(subset_idx)} samples)")
    print(f"{'#'*70}")

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    folds = list(skf.split(np.zeros(len(subset_idx)), subset_targets))

    frac_results = {}
    for strategy_name in strategies_to_run:
        ppath = partial_path(frac, strategy_name)
        if os.path.exists(ppath):
            with open(ppath, 'rb') as f:
                frac_results[strategy_name] = pickle.load(f)
            print(f"  [SKIP] {frac_label} | {strategy_name} "
                  f"(F1={np.mean(frac_results[strategy_name]['best_f1s']):.2f}%)")
            continue

        cfg = STRATEGIES[strategy_name]
        h_scores = HARDNESS_SCORES[cfg['hardness']]

        print(f"\n{'='*60}")
        print(f"  {frac_label} | {strategy_name}")
        print(f"{'='*60}")

        strat = {'best_f1s': [], 'last_f1s': [], 'last_accs': [], 'histories': [],
                 'best_epochs': [], 'best_times': [], 'total_times': [],
                 'last_per_class': [], 'last_cms': [], 'saved_paths': []}

        for fold_idx, (f_train, f_test) in enumerate(folds):
            res = train_fold(strategy_name, h_scores, cfg['direction'],
                             subset_idx[f_train], subset_idx[f_test], subset_idx,
                             fold_idx + 1, frac, frac_label)
            strat['best_f1s'].append(res['best_f1'])
            strat['last_f1s'].append(res['last_f1'])
            strat['last_accs'].append(res['last_acc'])
            strat['histories'].append(res['history'])
            strat['best_epochs'].append(res['best_epoch'])
            strat['best_times'].append(res['best_time'])
            strat['total_times'].append(res['total_time'])
            strat['last_per_class'].append(res['last_per_class'])
            strat['last_cms'].append(res['last_cm'])
            strat['saved_paths'].append(res['saved_paths'])

        with open(ppath, 'wb') as f:
            pickle.dump(strat, f)

        print(f"  [{frac_label}] [{strategy_name}] Best F1="
              f"{np.mean(strat['best_f1s']):.2f}+/-{np.std(strat['best_f1s']):.2f}%")
        frac_results[strategy_name] = strat

    all_fraction_results[frac] = frac_results

print(f"\nAll done in {(time.time()-grand_start)/3600:.1f} hours")

In [ ]:
pkl_path = os.path.join(REPORT_DIR, 'histories', 'datasize_ablation.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(all_fraction_results, f)

rows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        for fi, hist in enumerate(strat['histories']):
            for ep in range(len(hist['train_loss'])):
                rows.append({
                    'competence_fn': COMPETENCE_FN,
                    'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
                    'n_total_samples': len(SUBSETS[frac]),
                    'strategy': strat_name, 'hardness': cfg['hardness'],
                    'direction': cfg['direction'],
                    'fold': fi + 1, 'epoch': ep + 1,
                    'train_loss': hist['train_loss'][ep], 'train_f1': hist['train_f1'][ep],
                    'val_loss': hist['val_loss'][ep], 'val_f1': hist['val_f1'][ep],
                    'val_acc': hist['val_acc'][ep], 'samples_used': hist['samples_used'][ep],
                })
epoch_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_epoch_data.csv')
pd.DataFrame(rows).to_csv(epoch_csv, index=False)

srows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        for fi in range(len(strat['best_f1s'])):
            pc = strat['last_per_class'][fi]
            srows.append({
                'competence_fn': COMPETENCE_FN,
                'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
                'n_samples': len(SUBSETS[frac]),
                'strategy': strat_name, 'hardness': cfg['hardness'],
                'direction': cfg['direction'], 'fold': fi + 1,
                'best_f1': strat['best_f1s'][fi], 'best_epoch': strat['best_epochs'][fi],
                'time_to_best_s': strat['best_times'][fi],
                'last_f1': strat['last_f1s'][fi], 'last_acc': strat['last_accs'][fi],
                'total_time_s': strat['total_times'][fi],
                'f1_adenomatous': pc[0], 'f1_cancer': pc[1],
                'f1_inflamed': pc[2], 'f1_normal': pc[3],
            })
fold_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_per_fold.csv')
pd.DataFrame(srows).to_csv(fold_csv, index=False)

arows = []
for frac, frac_res in all_fraction_results.items():
    for strat_name, strat in frac_res.items():
        cfg = STRATEGIES[strat_name]
        arows.append({
            'competence_fn': COMPETENCE_FN,
            'fraction': frac, 'fraction_label': f"{frac*100:.0f}%",
            'n_samples': len(SUBSETS[frac]),
            'strategy': strat_name, 'hardness': cfg['hardness'], 'direction': cfg['direction'],
            'best_f1_mean': np.mean(strat['best_f1s']), 'best_f1_std': np.std(strat['best_f1s']),
            'last_f1_mean': np.mean(strat['last_f1s']), 'last_f1_std': np.std(strat['last_f1s']),
            'avg_best_epoch': np.mean(strat['best_epochs']),
            'avg_time_to_best': np.mean(strat['best_times']),
            'avg_total_time': np.mean(strat['total_times']),
        })
summary_csv = os.path.join(REPORT_DIR, 'histories', 'datasize_summary.csv')
pd.DataFrame(arows).to_csv(summary_csv, index=False)

print(f"Saved: {pkl_path}")
print(f"Saved: {epoch_csv}")
print(f"Saved: {fold_csv}")
print(f"Saved: {summary_csv}")

In [ ]:
print(f"{'='*100}")
print(f"  DATA SIZE ABLATION - Best Model Macro F1  ({COMPETENCE_FN})")
print(f"{'='*100}")

header = f"{'Strategy':<16}"
for frac in DATA_FRACTIONS:
    header += f"{f'{frac*100:.0f}%':>18}"
print(header)
print('-' * 100)

for strat_name in strategies_to_run:
    line = f"{strat_name:<16}"
    for frac in DATA_FRACTIONS:
        s = all_fraction_results[frac][strat_name]
        cell = f"{np.mean(s['best_f1s']):.2f}+/-{np.std(s['best_f1s']):.2f}"
        line += f"{cell:>18}"
    print(line)
print(f"{'='*100}")

for frac in DATA_FRACTIONS:
    fr = all_fraction_results[frac]
    means = {k: np.mean(v['best_f1s']) for k, v in fr.items()}
    winner = max(means, key=means.get)
    rnd = means.get('Random', np.nan)
    print(f"  {frac*100:>3.0f}%: {winner:<16} {means[winner]:.2f}%   "
          f"(Random {rnd:.2f}%, margin {means[winner]-rnd:+.2f})")

In [ ]:
cb_colors = {
    'Loss_E2H':     '#0072B2', 'Loss_H2E':     '#56B4E9',
    'Boundary_E2H': '#D55E00', 'Boundary_H2E': '#E69F00',
    'Centroid_E2H': '#009E73', 'Centroid_H2E': '#66CC99',
    'Random':       '#000000',
}
cb_ls = {k: ('-' if k.endswith('E2H') else ('--' if k.endswith('H2E') else ':')) for k in cb_colors}
cb_mk = {k: ('o' if k.endswith('E2H') else ('s' if k.endswith('H2E') else 'D')) for k in cb_colors}

x = np.arange(len(DATA_FRACTIONS))
x_labels = [f"{f*100:.0f}%\n({len(SUBSETS[f])})" for f in DATA_FRACTIONS]

fig, ax = plt.subplots(figsize=(12, 7))
for strat_name in strategies_to_run:
    means = [np.mean(all_fraction_results[f][strat_name]['best_f1s']) for f in DATA_FRACTIONS]
    stds = [np.std(all_fraction_results[f][strat_name]['best_f1s']) for f in DATA_FRACTIONS]
    lw = 3 if strat_name == 'Random' else 2
    ax.errorbar(x, means, yerr=stds, marker=cb_mk[strat_name], ms=8, lw=lw, capsize=4,
                color=cb_colors[strat_name], ls=cb_ls[strat_name], label=strat_name)
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Training Data Fraction (n samples)', fontsize=13)
ax.set_ylabel('Best Macro F1 (%)', fontsize=13)
ax.set_title('Curriculum Learning Effect vs Data Size', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'f1_vs_datasize.png'), dpi=150, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, 6))
for strat_name in strategies_to_run:
    if strat_name == 'Random':
        continue
    margins = [np.mean(all_fraction_results[f][strat_name]['best_f1s']) -
               np.mean(all_fraction_results[f]['Random']['best_f1s']) for f in DATA_FRACTIONS]
    ax.plot(x, margins, marker=cb_mk[strat_name], ms=8, lw=2,
            color=cb_colors[strat_name], ls=cb_ls[strat_name], label=strat_name)
ax.axhline(0, color='black', lw=1.5, ls=':', label='Random baseline')
ax.set_xticks(x)
ax.set_xticklabels(x_labels, fontsize=11)
ax.set_xlabel('Training Data Fraction (n samples)', fontsize=13)
ax.set_ylabel('Macro F1 margin over Random (pp)', fontsize=13)
ax.set_title('Curriculum Benefit Relative to Random Ordering', fontsize=15, fontweight='bold')
ax.legend(fontsize=10, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'margin_over_random.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df = pd.read_csv(os.path.join(REPORT_DIR, 'histories', 'datasize_epoch_data.csv'))
epochs = np.arange(1, NUM_EPOCHS + 1)
n_frac = len(DATA_FRACTIONS)

fig, axes = plt.subplots(1, n_frac, figsize=(6.5 * n_frac, 6), sharey=True)
if n_frac == 1:
    axes = [axes]

for idx, frac in enumerate(DATA_FRACTIONS):
    ax = axes[idx]
    sub = df[df['fraction'] == frac]
    for strat in strategies_to_run:
        sd = sub[sub['strategy'] == strat]
        if sd.empty:
            continue
        agg = sd.groupby('epoch')['val_f1'].agg(['mean', 'std'])
        lw = 3 if strat == 'Random' else 2
        ax.plot(epochs, agg['mean'], color=cb_colors[strat], ls=cb_ls[strat], lw=lw, label=strat)
        ax.fill_between(epochs, agg['mean'] - agg['std'], agg['mean'] + agg['std'],
                        color=cb_colors[strat], alpha=0.06)
    ax.set_title(f"{frac*100:.0f}% ({len(SUBSETS[frac])} samples)", fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=12)
    if idx == 0:
        ax.set_ylabel('Validation Macro F1 (%)', fontsize=12)
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('Validation F1 Curves - All Strategies x Data Sizes', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'val_f1_all_strategies_all_fractions.png'),
            dpi=150, bbox_inches='tight')
plt.show()